In [0]:
from pyspark.sql import functions as F

In [0]:
delitos = spark.table("delitos.default.da_carpetas_de_investigacion_pgj_cdmx")

In [0]:


# Convierte fechas problemáticas a string primero, limpia NAs, luego castea
delitos =delitos \
    .withColumn("FechaHecho", 
        F.when(F.col("FechaHecho") == "NA", None)
        .otherwise(F.col("FechaHecho").cast("string"))) \
    .withColumn("FechaInicio",
        F.col("FechaInicio").cast("string")) \
    .withColumn("FechaHecho", F.to_date(F.col("FechaHecho"), "yyyy-MM-dd")) \
    .withColumn("FechaInicio", F.to_date(F.col("FechaInicio"), "yyyy-MM-dd")) \
    .filter(F.col("FechaHecho").isNotNull()) \
    .filter(F.col("FechaInicio").isNotNull())

In [0]:
delitos.count()
delitos.printSchema()
display(delitos)
delitos.createOrReplaceTempView("delitos")
spark.sql("select * from delitos").count()


In [0]:


delitos.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in delitos.columns
]).show()

In [0]:
total = delitos.count()

nulos_colonia = delitos.filter(F.col("municipio_hechos")=="NA").count()
print(f"Nulos: {nulos_colonia} ({round(nulos_colonia/total*100, 2)}%)")

In [0]:
delitos.filter(F.col("municipio_hechos")=="NA") \
  .select("municipio_hechos", "longitud", "latitud") \
  .show(10)

In [0]:
delitos.groupBy("municipio_hechos") \
  .count() \
  .orderBy(F.col("count").desc()) \
  .show(20)

In [0]:
delitos.filter(F.col("municipio_hechos") == "NA") \
  .groupBy("AlcaldiaHechos") \
  .count() \
  .orderBy(F.col("count").desc()) \
  .show(20)

In [0]:
delitos.select("AlcaldiaHechos") \
  .distinct() \
  .filter(F.col("AlcaldiaHechos").startswith("CUAJIMALPA") | 
          F.col("AlcaldiaHechos").startswith("LA MAGDALENA") |
          F.col("AlcaldiaHechos").startswith("GUSTAVO")) \
  .show(truncate=False)

In [0]:
df_clean = delitos.withColumn(
    "AlcaldiaHechos",
    F.regexp_replace(F.col("AlcaldiaHechos"), "GUSTAVO A\\. MADERO", "GUSTAVO A MADERO")
)

In [0]:
alcaldias_cdmx = [
    "CUAUHTEMOC", "IZTAPALAPA", "BENITO JUAREZ", "COYOACAN",
    "MIGUEL HIDALGO", "ALVARO OBREGON", "VENUSTIANO CARRANZA",
    "TLALPAN", "AZCAPOTZALCO", "IZTACALCO", "XOCHIMILCO",
    "TLAHUAC", "LA MAGDALENA CONTRERAS", "CUAJIMALPA DE MORELOS",
    "MILPA ALTA", "GUSTAVO A MADERO"
]

df_clean = df_clean.filter(F.col("AlcaldiaHechos").isin(alcaldias_cdmx))

In [0]:
df_clean = df_clean.drop("municipio_hechos", "colonia_datos", "fgj_colonia_registro")

In [0]:
# print(f"Registros originales: {delitos.count()}")
print(f"Registros limpios: {df_clean.count()}")

In [0]:
df_clean.select("ao_hechos", "mes_hechos", "FechaHecho", "HoraHecho") \
    .show(10, truncate=False)

In [0]:
df_clean.select("FechaHecho").distinct().orderBy("FechaHecho").show(20, truncate=False)

In [0]:
df_clean.filter(F.col("FechaHecho") < "2016-01-01") \
    .select("FechaHecho", "FechaInicio") \
    .show(10, truncate=False)

In [0]:
df_clean.select("mes_hechos").distinct().orderBy("mes_hechos").show(20, truncate=False)

In [0]:
meses = {
    "Enero": 1, "Febrero": 2, "Marzo": 3, "Abril": 4,
    "Mayo": 5, "Junio": 6, "Julio": 7, "Agosto": 8,
    "Septiembre": 9, "Octubre": 10, "Noviembre": 11, "Diciembre": 12
}

mapping_expr = F.create_map([F.lit(x) for kv in meses.items() for x in kv])

df_clean = df_clean.withColumn(
    "mes_hechos_num",
    mapping_expr[F.col("mes_hechos")]
)

In [0]:
display(df_clean)

In [0]:
df_clean.select("mes_hechos", "mes_hechos_num").distinct().orderBy("mes_hechos_num").show()

In [0]:
df_clean.filter(F.col("mes_hechos") == "NA").count()


In [0]:
df_clean.select("categoria_delito").distinct().orderBy("categoria_delito").show(30, truncate=False)

In [0]:
df_clean.select("delito").distinct().orderBy("delito").count()

In [0]:
for col in ["categoria_delito", "fiscalia", "agencia", "unidad_investigacion"]:
    count = df_clean.filter(
        F.col(col).isNull() | (F.col(col) == "NA") | (F.col(col) == "SIN DATO")
    ).count()
    print(f"--- {col}: {count} ---")

In [0]:
df_clean.select("fiscalia").distinct().count()

In [0]:
df_clean = df_clean.drop("unidad_investigacion", "agencia")

In [0]:
df_clean.printSchema()

In [0]:
df_clean = df_clean \
    .withColumn("HoraHecho", F.to_timestamp(F.col("HoraHecho"), "HH:mm:ss")) \
    .withColumn("ao_hechos", F.col("ao_hechos").cast("integer")) \
    .withColumn("hora_del_dia", F.hour(F.col("HoraHecho"))) \
    .withColumn("dia_semana", F.dayofweek(F.col("FechaHecho"))) \
    .withColumn("dia_mes", F.dayofmonth(F.col("FechaHecho"))) \
    .withColumn("trimestre", F.quarter(F.col("FechaHecho")))

In [0]:
df_clean = df_clean \
    .withColumn("ao_inicio", F.col("ao_inicio").cast("integer"))

In [0]:
df_clean.printSchema()

In [0]:
df_clean = df_clean.filter(df_clean["FechaHecho"] >= "2017-01-01")

In [0]:
df_clean = df_clean \
    .withColumn("FechaHecho", 
        F.try_to_date(F.col("FechaHecho").cast("string"), "yyyy-MM-dd")) \
    .withColumn("FechaInicio", 
        F.try_to_date(F.col("FechaInicio").cast("string"), "yyyy-MM-dd"))

In [0]:
df_clean.select("FechaHecho") \
  .groupBy("FechaHecho") \
  .count() \
  .orderBy(F.col("count").asc()) \
  .show(20)

In [0]:
df_clean = df_clean.filter(F.col("mes_hechos_num").isNotNull())


In [0]:
df_clean.printSchema()

In [0]:
df_clean = df_clean \
    .withColumn("hora_inicio", F.hour(F.col("HoraInicio"))) \
    .drop("mes_hechos", "HoraHecho", "HoraInicio")

In [0]:
df_clean = df_clean.withColumn(
    "dias_para_registro",
    F.datediff(F.col("FechaInicio"), F.col("FechaHecho"))
)

In [0]:
df_clean.printSchema()
df_clean.select("dias_para_registro").describe().show()


In [0]:
df_clean = df_clean.filter(F.col("mes_inicio")!="NA")

In [0]:
df_clean = df_clean.withColumn(
    "mes_inicio_num",
    mapping_expr[F.col("mes_inicio")]
).drop("mes_inicio")

In [0]:
df_clean.withColumn("dias_para_registro", F.col("dias_para_registro").cast("int")).filter(F.col("dias_para_registro") < 0).count()
df_clean.withColumn("dias_para_registro", F.col("dias_para_registro").cast("int")).filter(F.col("dias_para_registro") > 3650).count()

In [0]:
df_clean.select(F.col("dias_para_registro")).groupBy("dias_para_registro").count().show()


In [0]:
negativos = df_clean.filter(F.col("dias_para_registro") < 0).count()
extremos = df_clean.filter(F.col("dias_para_registro") > 3650).count()

print(f"Negativos: {negativos}")
print(f"Mayores a 10 años: {extremos}")

In [0]:
df_clean = df_clean.filter(F.col("dias_para_registro") >= 0)

In [0]:
df_clean.show()

In [0]:
df_clean.count()

In [0]:
df_clean.select(F.min("FechaHecho"), F.max("FechaHecho")).show()

In [0]:
df_clean.printSchema()

In [0]:
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("delitos.silver.delitos_cdmx_silver")

In [0]:
df_verify = spark.table("delitos.silver.delitos_cdmx_silver")
print(f"Registros: {df_verify.count()}")
df_verify.printSchema()